In [3]:
import cvxpy as cp
import numpy as np

# Problem parameters
n_banks = 3
n_months = 6
initial_balances = [10000, 15000, 5000]
interest_rates = [0.005, 0.010, 0.015]
monthly_budget = 6000

# Decision variables
x = cp.Variable((n_banks, n_months), nonneg=True)  # payments
b = cp.Variable((n_banks, n_months + 1), nonneg=True)  # balances

# Objective: minimize total payments
objective = cp.Minimize(cp.sum(x))

# Constraints
constraints = []

# Initial balance constraints
for i in range(n_banks):
    constraints.append(b[i, 0] == initial_balances[i])

# Balance update constraints
for i in range(n_banks):
    for j in range(n_months):
        constraints.append(
            b[i, j+1] == b[i, j] * (1 + interest_rates[i]) - x[i, j]
        )

# Monthly budget constraints
for j in range(n_months):
    constraints.append(cp.sum(x[:, j]) <= monthly_budget)

# Final payoff constraints
for i in range(n_banks):
    constraints.append(b[i, n_months] <= 0)

# Create and solve the problem
problem = cp.Problem(objective, constraints)
optimal_value = problem.solve()

print(f"Optimal total payment: ${optimal_value:,.2f}")

Optimal total payment: $30,713.87


In [4]:

# Display results
print(f"Optimal total payment: ${optimal_value:,.2f}")
print("\nPayment Schedule:")
print("Month | BOCHK | HSBC | SC | Total")
print("-" * 40)
for j in range(n_months):
    payments = [x[i, j].value for i in range(n_banks)]
    print(f"{j+1:5d} | ${payments[0]:6,.0f} | ${payments[1]:5,.0f} | "
          f"${payments[2]:4,.0f} | ${sum(payments):5,.0f}")

print("\nBalance Evolution:")
print("Month | BOCHK | HSBC | SC")
print("-" * 35)
for j in range(n_months + 1):
    balances = [b[i, j].value for i in range(n_banks)]
    print(f"{j:5d} | ${balances[0]:6,.0f} | ${balances[1]:5,.0f} | ${balances[2]:4,.0f}")

Optimal total payment: $30,713.87

Payment Schedule:
Month | BOCHK | HSBC | SC | Total
----------------------------------------
    1 | $     0 | $  925 | $5,075 | $6,000
    2 | $     0 | $6,000 | $   0 | $6,000
    3 | $     0 | $6,000 | $   0 | $6,000
    4 | $ 3,525 | $2,475 | $   0 | $6,000
    5 | $ 6,000 | $    0 | $   0 | $6,000
    6 | $   714 | $    0 | $   0 | $  714

Balance Evolution:
Month | BOCHK | HSBC | SC
-----------------------------------
    0 | $10,000 | $15,000 | $5,000
    1 | $10,050 | $14,225 | $   0
    2 | $10,100 | $8,367 | $   0
    3 | $10,151 | $2,451 | $   0
    4 | $ 6,677 | $    0 | $   0
    5 | $   710 | $    0 | $   0
    6 | $     0 | $    0 | $   0
